<a href="https://colab.research.google.com/github/MateoGlz/AdmiTareas-BackEnd/blob/main/6_Librerias_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import pandas as pd
import numpy as np
ruta = '/content/drive/MyDrive/Colab_Notebooks/Titanic-Dataset.csv'
df = pd.read_csv(ruta)

supervivencia_por_grupo = df.groupby(['Pclass', 'Sex'])['Survived'].agg([
    ('Total', 'count'),
    ('Supervivientes', 'sum'),
    ('Tasa_Supervivencia', lambda x: x.mean() * 100)
]).round(2)
max_tasa = supervivencia_por_grupo['Tasa_Supervivencia'].max()
grupo_max = supervivencia_por_grupo[supervivencia_por_grupo['Tasa_Supervivencia'] == max_tasa]

# Encontrar el grupo con la tasa mínima
min_tasa = supervivencia_por_grupo['Tasa_Supervivencia'].min()
grupo_min = supervivencia_por_grupo[supervivencia_por_grupo['Tasa_Supervivencia'] == min_tasa]

print(supervivencia_por_grupo)


print(f"\nMayor supervivencia: Clase {int(grupo_max.index[0][0])}, {grupo_max.index[0][1]} - {max_tasa}%")
print(f"Menor supervivencia: Clase {int(grupo_min.index[0][0])}, {grupo_min.index[0][1]} - {min_tasa}%")

               Total  Supervivientes  Tasa_Supervivencia
Pclass Sex                                              
1      female     94              91               96.81
       male      122              45               36.89
2      female     76              70               92.11
       male      108              17               15.74
3      female    144              72               50.00
       male      347              47               13.54

Mayor supervivencia: Clase 1 - 96.81%
Menor supervivencia: Clase 3, male - 13.54%


In [35]:
df['FamilySize'] = df['SibSp'] + df['Parch']
familias_grandes = df[df['FamilySize'] > 3]
num_familias_grandes = len(familias_grandes)
supervivencia_familias_grandes = familias_grandes['Survived'].mean() * 100
print(f"Número de pasajeros en familias grandes: {num_familias_grandes}")
print(f"Proporción de supervivencia en familias grandes: {supervivencia_familias_grandes:.2f}%")


Número de pasajeros en familias grandes: 62
Proporción de supervivencia en familias grandes: 16.13%


In [37]:
def clasificar_edad(edad):
    if pd.isna(edad):  # Manejar valores faltantes
        return 'Desconocido'
    elif edad < 18:
        return 'Menor de Edad'
    else:
        return 'Mayor de Edad'


df['MayoriaEdad'] = df['Age'].apply(clasificar_edad)

print("Distribucion por grupos de edad:")
print(df['MayoriaEdad'].value_counts())

print("\nProporción de supervivencia por grupo de edad:")
print(df.groupby('MayoriaEdad')['Survived'].mean() * 100)

Distribución por grupos de edad:
GrupoEdad
Mayor de Edad    601
Desconocido      177
Menor de Edad    113
Name: count, dtype: int64

Proporción de supervivencia por grupo de edad:
GrupoEdad
Desconocido      29.378531
Mayor de Edad    38.103161
Menor de Edad    53.982301
Name: Survived, dtype: float64


In [41]:

age_mean_numpy = np.nanmean(df['Age'].values)
fare_mean_numpy = np.nanmean(df['Fare'].values)

age_mean_pandas = df['Age'].mean()
fare_mean_pandas = df['Fare'].mean()

print("=== COMPARACIÓN DE PROMEDIOS ===")
print(f"Promedio de Age con NumPy:  {age_mean_numpy:.3f}")
print(f"Promedio de Age con Pandas: {age_mean_pandas:.3f}")
print(f"¿Son iguales? {np.isclose(age_mean_numpy, age_mean_pandas)}")
print()

print(f"Promedio de Fare con NumPy:  {fare_mean_numpy:.3f}")
print(f"Promedio de Fare con Pandas: {fare_mean_pandas:.3f}")
print(f"¿Son iguales? {np.isclose(fare_mean_numpy, fare_mean_pandas)}")
print()

print("=== VERIFICACIÓN DETALLADA ===")
print(f"Diferencia en Age: {abs(age_mean_numpy - age_mean_pandas):.10f}")
print(f"Diferencia en Fare: {abs(fare_mean_numpy - fare_mean_pandas):.10f}")

print(f"\nValores nulos en Age: {df['Age'].isnull().sum()}")
print(f"Valores nulos en Fare: {df['Fare'].isnull().sum()}")

=== COMPARACIÓN DE PROMEDIOS ===
Promedio de Age con NumPy:  29.699
Promedio de Age con Pandas: 29.699
¿Son iguales? True

Promedio de Fare con NumPy:  32.204
Promedio de Fare con Pandas: 32.204
¿Son iguales? True

=== VERIFICACIÓN DETALLADA ===
Diferencia en Age: 0.0000000000
Diferencia en Fare: 0.0000000000

Valores nulos en Age: 177
Valores nulos en Fare: 0


In [53]:
import numpy as np
import pandas as pd

fare_min = df['Fare'].min()
fare_max = df['Fare'].max()
intervalos = np.linspace(fare_min, fare_max, 6)

etiquetas = [f'Intervalo {i+1}' for i in range(len(intervalos)-1)]

df['IntervaloFare'] = pd.cut(df['Fare'], bins=intervalos, labels=etiquetas, include_lowest=True)

pasajeros_por_intervalo = df['IntervaloFare'].value_counts().sort_index()
supervivencia_por_intervalo = df.groupby('IntervaloFare', observed=False)['Survived'].mean() * 100

print("=== ANÁLISIS POR INTERVALOS DE TARIFA ===")
print("\nPasajeros por intervalo:")
for intervalo, cantidad in pasajeros_por_intervalo.items():
    print(f"{intervalo}: {cantidad} pasajeros")

print("\nProporción de supervivencia por intervalo:")
for intervalo, porcentaje in supervivencia_por_intervalo.items():
    print(f"{intervalo}: {porcentaje:.2f}%")

=== ANÁLISIS POR INTERVALOS DE TARIFA ===

Pasajeros por intervalo:
Intervalo 1: 838 pasajeros
Intervalo 2: 33 pasajeros
Intervalo 3: 17 pasajeros
Intervalo 4: 0 pasajeros
Intervalo 5: 3 pasajeros

Proporción de supervivencia por intervalo:
Intervalo 1: 36.16%
Intervalo 2: 75.76%
Intervalo 3: 64.71%
Intervalo 4: nan%
Intervalo 5: 100.00%


In [ ]:
print('Cantidad de Pasajeros por sexo:')
print(df['Sex'].value_counts())

Cantidad de Pasajeros por sexo:
Sex
male      577
female    314
Name: count, dtype: int64


In [ ]:
df['Sex'].groupby(df['Pclass']).value_counts().unstack()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
hombres = df[df['Sex'] == 'male']
mujeres = df[df['Sex'] == 'female']
hombres_sobrevivientes = hombres[hombres['Survived'] == 1]
mujeres_sobrevivientes = mujeres[mujeres['Survived'] == 1]
print(hombres_sobrevivientes.head(5))
print(mujeres_sobrevivientes.head(5))

    PassengerId  Survived  Pclass                          Name   Sex   Age  \
17           18         1       2  Williams, Mr. Charles Eugene  male   NaN   
21           22         1       2         Beesley, Mr. Lawrence  male  34.0   
23           24         1       1  Sloper, Mr. William Thompson  male  28.0   
36           37         1       3              Mamee, Mr. Hanna  male   NaN   
55           56         1       1             Woolner, Mr. Hugh  male   NaN   

    SibSp  Parch  Ticket     Fare Cabin Embarked  
17      0      0  244373  13.0000   NaN        S  
21      0      0  248698  13.0000   D56        S  
23      0      0  113788  35.5000    A6        S  
36      0      0    2677   7.2292   NaN        C  
55      0      0   19947  35.5000   C52        S  
   PassengerId  Survived  Pclass  \
1            2         1       1   
2            3         1       3   
3            4         1       1   
8            9         1       3   
9           10         1       2   

  

In [ ]:
sv = df[df['Survived'] == 1]
print(sv.head(5))

In [ ]:
sv = df[(df['Survived'] == 1) & (df['Pclass'])]
print(sv.head(5).group_by('Pclass'))


AttributeError: 'DataFrame' object has no attribute 'group_by'

In [ ]:

sv = df[df['Survived'] == 1]

for clase, grupo in sv.groupby('Pclass'):
    print(f"\nPclass {clase}")
    print(grupo.head(5))



Pclass 1
    PassengerId  Survived  Pclass  \
1             2         1       1   
3             4         1       1   
11           12         1       1   
23           24         1       1   
31           32         1       1   

                                                 Name     Sex   Age  SibSp  \
1   Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
3        Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
11                           Bonnell, Miss. Elizabeth  female  58.0      0   
23                       Sloper, Mr. William Thompson    male  28.0      0   
31     Spencer, Mrs. William Augustus (Marie Eugenie)  female   NaN      1   

    Parch    Ticket      Fare Cabin Embarked  
1       0  PC 17599   71.2833   C85        C  
3       0    113803   53.1000  C123        S  
11      0    113783   26.5500  C103        S  
23      0    113788   35.5000    A6        S  
31      0  PC 17569  146.5208   B78        C  

Pclass 2
    Pa